In [ ]:
import os
import gensim.downloader as api

'''
THE PACKAGES WILL GET DOWNLOADED TO C, WHICH I DONT WANT
'''
prjctDrctry = os.getcwd()
gensimCacheLoctn = os.path.join(prjctDrctry, "models")
os.environ["GENSIM_DATA_DIR"] = gensimCacheLoctn

print(f"Location : {gensimCacheLoctn}")
model = api.load("glove-wiki-gigaword-50")
print("Done!")

Location : D:\IIIT B\PersonalProjects\SemanticSearch\models


In [6]:
import numpy as np

In [11]:
def getRltdWrds(query, embdModel, topK=3):
    '''The idea is to expand the user query by words that are closer to the words in OUR CATALOG'''
    words = []

    qrySplit = query.split()
    
    for w in qrySplit:
        lwrWrd = w.lower()
        
        if lwrWrd in embdModel:
            words.append(lwrWrd)

    if not words:
        return query.split()

    smlrWrds = embdModel.most_similar(positive=words, topn=topK) #Gett similar from glove

    '''Combine original words with the discovered semantic neighbors'''
    fullContext = words + [word for word, score in smlrWrds]
    
    return list(set(fullContext))  #TO remove duplicates

In [ ]:
def getCtlgScore(fullContext, itemTxt, embdModel):
    """Scores catalog items based on how many semantic keywords match thgem"""
    words = [w.lower() for w in itemTxt.split()]
    score = 0

    for qWord in fullContext:

        if qWord in words: #Direct hit!!
            score += 1.0
            
        else:
            #Checks if anything from the catalog matches with query words
            for item in words:
                
                if (qWord in embdModel and item in embdModel):
                    similarity = embdModel.similarity(qWord, item) # glove similarity score function

                    if similarity > 0.65: #Only counts strong semantic reltionships || Hard threshold
                        score += similarity * 0.5

    return score

In [21]:
def semanticSrch(query, catalog, embdModel, k=2):
    '''Search pipeline'''

    fullContext = getRltdWrds(query, embdModel) #similar to rag => Its basically query rewriting by injecting more context
    
    print(f"Query : '{query}'")
    print(f"Injected keywords: {fullContext}\n")

    #Scoring items
    results = []
    
    for itmKey, itmName in catalog.items():
        score = getCtlgScore(fullContext, itmName, embdModel)
        results.append((itmKey, itmName, score))

    # Get topK
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:k]

In [35]:
menu = {
    "Item1": "Double Cheeseburger with Fries",
    "Item2": "Organic Avocado and Grilled Chicken Salad",
    "Item3": "Whey Protein Berry Smoothie Bowl",
    "Item4": "Pepperoni Deep Dish Pizza",
    "Item5": "Greek Yogurt with Oats and Honey",
}

In [36]:
query = "fitness breakfast protein"
topK = semanticSrch(query, menu, model, k=5)

Query : 'fitness breakfast protein'
Injected keywords: ['meal', 'protein', 'diet', 'breakfast', 'fitness', 'meals']



In [37]:
topK

[('Item4', 'Pepperoni Deep Dish Pizza', 1.3466593623161316),
 ('Item2', 'Organic Avocado and Grilled Chicken Salad', 1.043490618467331),
 ('Item3', 'Whey Protein Berry Smoothie Bowl', 1.0),
 ('Item1', 'Double Cheeseburger with Fries', 0),
 ('Item5', 'Greek Yogurt with Oats and Honey', 0)]